# ContractScanner LLM Evaluation, MLflow Tracing, and ROI Comparison

This notebook adds the final LLM and evaluation layer for the ContractScanner AI Agent final project.

The previous notebook prepared the contract analysis examples by loading CUAD contract chunks, retrieving relevant clauses, checking scope, and assigning a basic risk level. This notebook builds on those saved outputs by:

- Running contract questions through GPT-4o and GPT-4o-mini
- Keeping the out-of-scope examples as graceful rejection cases
- Logging five end-to-end examples with MLflow
- Creating evaluation scores for each example
- Explaining how human review fits into the evaluation process
- Comparing GPT-4o and GPT-4o-mini from a business ROI perspective

The goal is to show not only that the agent works technically, but also that it can be evaluated and compared in a way that makes sense for an enterprise legal/compliance use case.

In [0]:
# Install the packages needed for this notebook.
# openai is used for GPT-4o and GPT-4o-mini calls.
# mlflow is used for trace/evaluation logging.
# pandas is used to load and organize the saved agent outputs.

%pip install -q openai mlflow pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Import the Python libraries used throughout this notebook.

import os
import pandas as pd
import time
import mlflow
from openai import OpenAI

In [0]:
# Check the current notebook working directory.
# This helps confirm that Databricks is looking in the correct repo folder.

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: /Workspace/Users/shifrin.robert@gmail.com/contractscanner-ai-agent/notebooks
Files here: ['03_llm_evaluation_and_roi.ipynb', 'agent_outputs', 'CUAD_Data_Pipeline.ipynb', '02_agent_and_tools.ipynb.ipynb', 'cuad_chunks.pkl', 'cuad_embeddings.npy', 'cuad_faiss_index.index', '.gitkeep', '01_data_pipeline.ipynb', '03_llm_evaluation_and_roi_recommented_updated.ipynb']


In [0]:
# These are the saved outputs from the previous agent/tool prep notebook.
# The first file contains the 3 contract questions with retrieved contract context.
# The second file contains the 2 graceful rejection examples.

prep_path = "agent_outputs/agent_context_prep_results.csv"
reject_path = "agent_outputs/rejection_examples.csv"

print("Prep exists:", os.path.exists(prep_path))
print("Reject exists:", os.path.exists(reject_path))

Prep exists: True
Reject exists: True


In [0]:
# Load the saved examples into pandas DataFrames so we can inspect and reuse them.

prep_df = pd.read_csv(prep_path)
reject_df = pd.read_csv(reject_path)

display(prep_df)
display(reject_df)

question,status,risk_level,latency_seconds,top_retrieved_clause
Does this contract contain risky termination clauses?,ready_for_llm,Medium,0.197,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive."
What indemnification obligations are present?,ready_for_llm,High,0.037,"The foregoing indemnification obligations are conditioned upon the indemnified party promptly notifying the indemnifying party in writing of the claim, suit or proceeding for which the indemnifying party is obligated under this Section 24, cooperating with, assisting and providing information to, the indemnifying party as reasonably required, and granting the indemnifying party the exclusive right to defend or settle such claim, suit or proceeding."
Summarize the key risks in this agreement.,ready_for_llm,High,0.117,"BEEN ADVISED OF THE POSSIBILITY OR PROBABILITY OF SUCH DAMAGES AND EVEN IF THE REMEDIES OTHERWISE PROVIDED BY THIS AGREEMENT FAIL OF THEIR ESSENTIAL PURPOSE. THE REMEDIES PROVIDED BY THIS AGREEMENT AND THE PROVISIONS OF THIS AGREEMENT ALLOCATE THE RISKS OF THIS AGREEMENT BETWEEN THE PARTIES, SOME OF WHICH MAY BE UNKNOWN OR UNDERMINABLE. THESE LIMITATIONS ARE A MATERIAL INDUCEMENT FOR THE PARTIES TO THIS AGREEMENT TO ENTER INTO THIS AGREEMENT, AND THE PARTIES TO THIS AGREEMENT HAVE RELIED UPON THESE PROVISIONS IN DETERMINING WHETHER OR NOT TO ENTER INTO THIS AGREEMENT."


question,status,answer,latency_seconds
Can you help me plan a trip to Italy?,rejected,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",0.0
Can you diagnose my chest pain?,rejected,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",0.0


In [0]:
# This function wraps the final ContractScanner agent workflow into one place.
# It checks whether the question is out of scope first, then either returns
# the prepared rejection answer or calls the LLM with retrieved contract context.


def contract_agent(question, retrieved_clause=None, risk_level=None, model_name="gpt-4o-mini"):
    """
    Full agent workflow:
    1. Checks whether the question is in scope.
    2. Uses retrieved contract context from the tools notebook.
    3. Uses risk level from the risk assessment tool.
    4. Calls the selected LLM to generate a final answer.
    """

    # If the question matches one of the saved rejection examples,
    # the agent stops early instead of sending it to the LLM.
    if question in reject_df["question"].values:
        rejection_answer = reject_df.loc[
            reject_df["question"] == question, "answer"
        ].iloc[0]

        return {
            "status": "rejected",
            "question": question,
            "model": "scope_rejection_tool",
            "answer": rejection_answer,
            "used_tools": ["scope_check_tool"]
        }

    # For in-scope contract questions, the agent uses the retrieved clause,
    # the risk level, and the selected model to generate the response.
    try:
        answer, latency = call_contract_llm(
            model_name=model_name,
            question=question,
            retrieved_clause=retrieved_clause,
            risk_level=risk_level
        )

        return {
            "status": "answered",
            "question": question,
            "model": model_name,
            "answer": answer,
            "latency_seconds": latency,
            "used_tools": [
                "contract_retrieval_tool",
                "risk_assessment_tool",
                "llm_response_generator"
            ]
        }

    # This keeps the notebook from failing silently if the API call or processing fails.
    except Exception as e:
        return {
            "status": "error",
            "question": question,
            "model": model_name,
            "answer": f"The agent could not complete the request because of an API or processing error: {str(e)}",
            "used_tools": [
                "contract_retrieval_tool",
                "risk_assessment_tool",
                "llm_response_generator"
            ]
        }

In [0]:
# Check which Databricks secret scopes are available in this workspace.
# This was used to confirm the correct scope name before pulling the OpenAI key.

dbutils.secrets.listScopes()

[SecretScope(name='contractscanner')]

In [0]:
# Confirm that the OpenAI API key exists inside the contractscanner secret scope.
# The secret value itself is not printed, which keeps the key protected.

dbutils.secrets.list("contractscanner")

[SecretMetadata(key='openai_api_key')]

In [0]:
# Test OpenAI API access using the secret stored in Databricks.
# This confirms that GPT-4o-mini can be called using the contractscanner secret scope
# without hardcoding the API key in the notebook.

from openai import OpenAI
import time

openai_api_key = dbutils.secrets.get(
    scope="contractscanner",
    key="openai_api_key"
)

client = OpenAI(api_key=openai_api_key)

start_time = time.time()

test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what an indemnification clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency = round(time.time() - start_time, 3)

print("Model worked.")
print("Latency:", latency, "seconds")
print("Answer:")
print(test_response.choices[0].message.content)

Model worked.
Latency: 5.743 seconds
Answer:
An indemnification clause is a provision in a contract that requires one party to compensate the other for certain damages or losses incurred.


Trace(trace_id=tr-5d2d7aa8f3069d8cd85ff8213968a3a3)

In [0]:
# Test GPT-4o access.
# This confirms that both required models are available before we build the full evaluation.

start_time = time.time()

test_response_4o = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what a termination clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency_4o = round(time.time() - start_time, 3)

print("GPT-4o worked.")
print("Latency:", latency_4o, "seconds")
print("Answer:")
print(test_response_4o.choices[0].message.content)

GPT-4o worked.
Latency: 0.942 seconds
Answer:
A termination clause is a provision in a contract that outlines the conditions under which the agreement can be ended by either party.


Trace(trace_id=tr-dd16fd88f5dfed4420d37e7cf1a58c06)

## Run GPT-4o and GPT-4o-mini on Contract Questions

This section sends the three prepared contract-analysis questions through the `contract_agent()` workflow for both GPT-4o and GPT-4o-mini.

The earlier notebook already retrieved the most relevant CUAD contract clause and assigned a basic risk level. In this notebook, the agent uses that retrieved context and risk level, calls the selected LLM, and records the model response, latency, status, and tools used.

In [0]:
# This function takes one contract question and its retrieved CUAD clause,
# then asks the selected LLM to answer using only that context.

def call_contract_llm(model_name, question, retrieved_clause, risk_level):
    system_prompt = """
You are ContractScanner, an AI contract analysis assistant for legal and compliance teams.

Your job is to analyze contract clauses using the provided retrieved contract context.
You must:
- Answer only based on the retrieved context.
- Identify possible risks clearly.
- Avoid giving final legal advice.
- Recommend human legal/compliance review when needed.
- Keep the answer professional and concise.
"""

    user_prompt = f"""
User question:
{question}

Retrieved contract clause:
{retrieved_clause}

Initial risk level from tool:
{risk_level}

Write a clear contract analysis response.
"""

    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=500
    )

    latency = round(time.time() - start_time, 3)
    answer = response.choices[0].message.content

    return answer, latency

In [0]:
# Run the in-scope contract questions through both GPT-4o and GPT-4o-mini.
# Instead of calling the LLM directly here, each question now goes through
# contract_agent(), so the output includes status and tool usage.

models_to_compare = ["gpt-4o", "gpt-4o-mini"]

llm_results = []

for _, row in prep_df.iterrows():

    question = row["question"]
    retrieved_clause = row["top_retrieved_clause"]
    risk_level = row["risk_level"]

    for model_name in models_to_compare:

        print(f"Running {model_name} for question: {question}")

        result = contract_agent(
            question=question,
            retrieved_clause=retrieved_clause,
            risk_level=risk_level,
            model_name=model_name
        )

        llm_results.append({
            "question": question,
            "model": result["model"],
            "risk_level": risk_level,
            "retrieved_clause": retrieved_clause,
            "answer": result["answer"],
            "latency_seconds": result.get("latency_seconds", None),
            "status": result["status"],
            "used_tools": ", ".join(result["used_tools"])
        })

llm_results_df = pd.DataFrame(llm_results)

display(llm_results_df)

Running gpt-4o for question: Does this contract contain risky termination clauses?
Running gpt-4o-mini for question: Does this contract contain risky termination clauses?
Running gpt-4o for question: What indemnification obligations are present?
Running gpt-4o-mini for question: What indemnification obligations are present?
Running gpt-4o for question: Summarize the key risks in this agreement.
Running gpt-4o-mini for question: Summarize the key risks in this agreement.


question,model,risk_level,retrieved_clause,answer,latency_seconds,status,used_tools
Does this contract contain risky termination clauses?,gpt-4o,Medium,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive.","The retrieved contract clause indicates that certain clauses will survive termination of the agreement, specifically Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, and 30-34. This means that the obligations or rights outlined in these clauses will continue to be in effect even after the contract is terminated. **Potential Risks:** 1. **Unclear Obligations:** Without specific details on what these clauses entail, there is a risk that ongoing obligations could be burdensome or unfavorable. 2. **Extended Liability:** Surviving clauses might extend liability or responsibilities beyond the termination, which could be risky if not clearly understood. 3. **Enforcement and Compliance:** Ensuring compliance with surviving clauses can be challenging, especially if they involve complex or ongoing commitments. **Recommendation:** - A detailed review of the specific clauses mentioned is necessary to assess the full scope and implications of their survival post-termination. - It is advisable to consult with a legal professional to evaluate the potential risks associated with these surviving clauses and to ensure that they align with the overall strategic and operational goals of the parties involved.",3.751,answered,"contract_retrieval_tool, risk_assessment_tool, llm_response_generator"
Does this contract contain risky termination clauses?,gpt-4o-mini,Medium,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive.","The retrieved contract clause indicates that several specific clauses will survive termination of the agreement, regardless of the cause. This can present potential risks, particularly if the surviving clauses impose ongoing obligations or liabilities on the parties after termination. Key considerations include: 1. **Ongoing Obligations**: The survival of certain clauses may require continued compliance or performance, which could lead to unexpected liabilities. 2. **Scope of Surviving Clauses**: Without knowing the content of the specific clauses listed (4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34), it is difficult to assess the full impact. Some clauses may pertain to confidentiality, indemnification, or dispute resolution, which could have significant implications post-termination. 3. **Lack of Clarity**: The phrase ""for whatever cause"" suggests that termination could occur under various circumstances, potentially leading to disputes regarding the interpretation of obligations that survive. Given these factors, the initial medium risk level appears warranted. It is advisable to conduct a thorough review of the specific clauses that survive termination to fully understand their implications. A human legal or compliance review is recommended to assess the potential risks associated with these termination clauses comprehensively.",3.827,answered,"contract_retrieval_tool, risk_assessment_tool, llm_response_generator"
What indemnification obligations are present?,gpt-4o,High,"The foregoing indemnification obligations are conditioned upon the indemnified party promptly notifying the indemnifying party in writing of the claim, suit or proceeding for which the indemnifying party is obligated under this Section 24, cooperating with, assisting and providing information to, the indemnifying party as reasonably required, and granting the indemnifying party the exclusive right to defend or settle such claim, suit or proceeding.","The indemnification obligations outlined in the contract require the indemnified party to: 1. Promptly notify the indemnifying party in writing of any claim, suit, or proceeding. 2. Cooperate with, assist, and provide necessary in

[Trace(trace_id=tr-e9f3eeb9d6a3927bebce933b09e1af68), Trace(trace_id=tr-b7da06b68286dd809fe73ab49603f787), Trace(trace_id=tr-ff5c04d2c3e8fd92a007ade2d950dfca), Trace(trace_id=tr-0cbad5f86af0e61a26f2ce100123a118), Trace(trace_id=tr-66384f017e94b0d25a10be1247b8dd91), Trace(trace_id=tr-184f50a9255d4aac893cad7f1845b58c)]

## Evaluation Scoring

This section creates simple evaluation scores for the GPT-4o and GPT-4o-mini contract responses.

The scoring focuses on:
- Relevance to the user question
- Grounding in the retrieved contract clause
- Risk identification
- Clarity
- Human review recommendation

This is not meant to be a perfect legal evaluation. It is a lightweight rubric to compare the two models in a consistent way for the project.

In [0]:
# Simple evaluation function for contract-analysis responses.

def evaluate_contract_response(question, retrieved_clause, risk_level, answer):
    answer_lower = answer.lower()
    question_lower = question.lower()
    clause_lower = retrieved_clause.lower()

    relevance_score = 5 if any(word in answer_lower for word in question_lower.split()[:4]) else 4

    grounding_score = 5 if any(
        keyword in answer_lower
        for keyword in clause_lower.split()[:12]
    ) else 4

    risk_score = 5 if "risk" in answer_lower or risk_level.lower() in answer_lower else 4

    clarity_score = 5 if len(answer.split()) >= 40 and len(answer.split()) <= 180 else 4

    human_review_score = 5 if "legal" in answer_lower or "review" in answer_lower or "counsel" in answer_lower else 4

    average_score = round(
        (
            relevance_score
            + grounding_score
            + risk_score
            + clarity_score
            + human_review_score
        ) / 5,
        2
    )

    return {
        "relevance_score": relevance_score,
        "grounding_score": grounding_score,
        "risk_score": risk_score,
        "clarity_score": clarity_score,
        "human_review_score": human_review_score,
        "average_score": average_score
    }

In [0]:
# Apply the evaluation function to each LLM response.

evaluation_rows = []

for _, row in llm_results_df.iterrows():
    scores = evaluate_contract_response(
        question=row["question"],
        retrieved_clause=row["retrieved_clause"],
        risk_level=row["risk_level"],
        answer=row["answer"]
    )

    evaluation_rows.append({
        "question": row["question"],
        "model": row["model"],
        "risk_level": row["risk_level"],
        "latency_seconds": row["latency_seconds"],
        "answer": row["answer"],
        **scores
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

question,model,risk_level,latency_seconds,answer,relevance_score,grounding_score,risk_score,clarity_score,human_review_score,average_score
Does this contract contain risky termination clauses?,gpt-4o,Medium,3.751,"The retrieved contract clause indicates that certain clauses will survive termination of the agreement, specifically Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, and 30-34. This means that the obligations or rights outlined in these clauses will continue to be in effect even after the contract is terminated. **Potential Risks:** 1. **Unclear Obligations:** Without specific details on what these clauses entail, there is a risk that ongoing obligations could be burdensome or unfavorable. 2. **Extended Liability:** Surviving clauses might extend liability or responsibilities beyond the termination, which could be risky if not clearly understood. 3. **Enforcement and Compliance:** Ensuring compliance with surviving clauses can be challenging, especially if they involve complex or ongoing commitments. **Recommendation:** - A detailed review of the specific clauses mentioned is necessary to assess the full scope and implications of their survival post-termination. - It is advisable to consult with a legal professional to evaluate the potential risks associated with these surviving clauses and to ensure that they align with the overall strategic and operational goals of the parties involved.",5,5,5,5,5,5.0
Does this contract contain risky termination clauses?,gpt-4o-mini,Medium,3.827,"The retrieved contract clause indicates that several specific clauses will survive termination of the agreement, regardless of the cause. This can present potential risks, particularly if the surviving clauses impose ongoing obligations or liabilities on the parties after termination. Key considerations include: 1. **Ongoing Obligations**: The survival of certain clauses may require continued compliance or performance, which could lead to unexpected liabilities. 2. **Scope of Surviving Clauses**: Without knowing the content of the specific clauses listed (4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34), it is difficult to assess the full impact. Some clauses may pertain to confidentiality, indemnification, or dispute resolution, which could have significant implications post-termination. 3. **Lack of Clarity**: The phrase ""for whatever cause"" suggests that termination could occur under various circumstances, potentially leading to disputes regarding the interpretation of obligations that survive. Given these factors, the initial medium risk level appears warranted. It is advisable to conduct a thorough review of the specific clauses that survive termination to fully understand their implications. A human legal or compliance review is recommended to assess the potential risks associated with these termination clauses comprehensively.",5,5,5,4,5,4.8
What indemnification obligations are present?,gpt-4o,High,3.185,"The indemnification obligations outlined in the contract require the indemnified party to: 1. Promptly notify the indemnifying party in writing of any claim, suit, or proceeding. 2. Cooperate with, assist, and provide necessary information to the indemnifying party. 3. Grant the indemnifying party the exclusive right to defend or settle the claim, suit, or proceeding. These obligations place significant responsibilities on the indemnified party to ensure that the indemnifying party is informed and able to manage the defense or settlement effectively. The initial risk level is considered high, likely due to the potential consequences of failing to meet these conditions, which could result in the indemnifying party refusing to cover the claim. It is recommended that a human legal or compliance professional review these obligations to assess the potential risks and ensure that the indemnified party is capable of fulfilling these requirements.",5,5,5,5,5,5.0
What indemnification obligations are present?,gpt-4o-mini,High,3.729,"The indemnification obl

## ROI Comparison

This section compares GPT-4o and GPT-4o-mini from a business perspective.

The goal is not only to see which model gives better answers, but also which model makes more sense for a real enterprise deployment. ContractScanner would save money by reducing analyst review time, but the model cost still matters if many contracts are processed.

The ROI calculation uses estimated token counts because this notebook uses the chat completions API and stores the model outputs, latency, and scores instead of storing exact token usage in the results table.

In [0]:
# ROI comparison for GPT-4o vs GPT-4o-mini.
# These prices are written as variables so they can be updated if OpenAI pricing changes.

# Current standard OpenAI API prices per 1M tokens.
# Source: OpenAI pricing page.
MODEL_PRICING = {
    "gpt-4o": {
        "input_cost_per_1m": 2.50,
        "output_cost_per_1m": 10.00
    },
    "gpt-4o-mini": {
        "input_cost_per_1m": 0.15,
        "output_cost_per_1m": 0.60
    }
}

# Business assumptions from the original project proposal.
contracts_per_year = 500
hours_saved_per_contract = 1
analyst_hourly_rate = 75

annual_labor_savings = contracts_per_year * hours_saved_per_contract * analyst_hourly_rate

roi_rows = []

for model_name in ["gpt-4o", "gpt-4o-mini"]:
    model_rows = llm_results_df[llm_results_df["model"] == model_name]

    # Because this notebook used chat.completions, token usage was not saved in llm_results_df.
    # For the ROI estimate, we use a conservative average token estimate per contract question.
    estimated_input_tokens_per_contract = 1000
    estimated_output_tokens_per_contract = 350

    input_cost = (
        estimated_input_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["input_cost_per_1m"]
    )

    output_cost = (
        estimated_output_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["output_cost_per_1m"]
    )

    estimated_cost_per_contract = input_cost + output_cost
    estimated_annual_llm_cost = estimated_cost_per_contract * contracts_per_year
    estimated_net_savings = annual_labor_savings - estimated_annual_llm_cost

    avg_latency = round(model_rows["latency_seconds"].mean(), 3)
    avg_score = round(evaluation_df[evaluation_df["model"] == model_name]["average_score"].mean(), 2)

    roi_rows.append({
        "model": model_name,
        "average_eval_score": avg_score,
        "average_latency_seconds": avg_latency,
        "estimated_cost_per_contract_usd": round(estimated_cost_per_contract, 6),
        "estimated_annual_llm_cost_usd": round(estimated_annual_llm_cost, 2),
        "annual_labor_savings_usd": annual_labor_savings,
        "estimated_net_savings_usd": round(estimated_net_savings, 2)
    })

roi_df = pd.DataFrame(roi_rows)

display(roi_df)

model,average_eval_score,average_latency_seconds,estimated_cost_per_contract_usd,estimated_annual_llm_cost_usd,annual_labor_savings_usd,estimated_net_savings_usd
gpt-4o,4.93,3.162,0.006,3.0,37500,37497.0
gpt-4o-mini,4.8,3.931,3.6E-4,0.18,37500,37499.82


In [0]:
# Run the out-of-scope examples through the same agent function.
# These are saved separately because they test rejection behavior,
# not grounded contract-answer quality.

rejection_agent_rows = []

for _, row in reject_df.iterrows():
    result = contract_agent(question=row["question"])

    rejection_agent_rows.append({
        "question": result["question"],
        "status": result["status"],
        "model": result["model"],
        "answer": result["answer"],
        "used_tools": ", ".join(result["used_tools"])
    })

rejection_agent_df = pd.DataFrame(rejection_agent_rows)

display(rejection_agent_df)

question,status,model,answer,used_tools
Can you help me plan a trip to Italy?,rejected,scope_rejection_tool,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",scope_check_tool
Can you diagnose my chest pain?,rejected,scope_rejection_tool,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",scope_check_tool


In [0]:
# Save the rejection examples separately from the model evaluation table.
# This keeps the contract-answer scores and the scope-rejection checks from being mixed together.

rejection_output_path = "agent_outputs/final_rejection_examples.csv"
rejection_agent_df.to_csv(rejection_output_path, index=False)

print("Saved rejection examples to:", rejection_output_path)

Saved rejection examples to: agent_outputs/final_rejection_examples.csv


## Deployment Recommendation

Based on the evaluation and ROI comparison, I would use GPT-4o-mini as the default model for ContractScanner. It is much cheaper than GPT-4o and still gave strong enough responses for the normal contract review questions. For a first-pass contract tool, that matters because the system could be used on a lot of contracts and the cost difference would add up fast.

I would still keep GPT-4o available as the escalation model. If the risk level is high, the question is unclear, or the GPT-4o-mini response gets a lower evaluation score, then it makes sense to send the same retrieved clause to GPT-4o for a stronger second review.

The updated agent flow also makes the deployment cleaner because it now returns a status and the tools used. That makes it easier to tell the difference between an answered contract question, a rejected out-of-scope question, and an API or processing error.

Overall, the best setup is a hybrid approach: GPT-4o-mini for normal questions, GPT-4o for higher-risk or unclear cases, and human legal/compliance review for anything that could have real legal impact.

## Final Notes and Deviations from Proposal

This section still follows the main project proposal, but the edited notebook makes the agent workflow more complete than my first version. Instead of only calling the LLM directly for contract questions, the notebook now uses a `contract_agent()` function that checks scope, uses the retrieved CUAD clause, uses the risk level, calls the selected model, and returns the final status and tools used.

One change from the original plan is that I still used prepared CSV outputs from the earlier agent/tool notebook instead of building a full production upload interface. I think that was the right tradeoff for this project because the main goal was to show the agent workflow, model comparison, tracing, evaluation, rejection handling, and ROI analysis.

Another change is how the rejection examples are handled. In the original version, the rejection rows were added into the final evaluation table. In the edited version, the out-of-scope questions are passed through the same agent function but saved separately as `final_rejection_examples.csv`. That makes more sense because those rows are testing the scope-checking behavior, not the quality of a grounded contract answer.

Overall, GPT-4o-mini is still my recommended default model because it gives the better cost-to-performance balance for normal contract review questions. GPT-4o should be kept for escalation when the question is high-risk, vague, or needs a stronger second pass. I would not treat either model as a replacement for legal review, but the workflow is useful as a first-pass contract analysis tool.